# 06h — Dense vs TF-IDF vs Hybrid Retrieval

**Notebook version: 06h-dense-hybrid-v7**

## 这一步通俗来说在做什么？

把知识库想成一个图书馆：

- **TF-IDF** 像按照书名中的相同词语找书。它擅长精确术语，例如 `Winter Fuel Payment`；
- **Dense Embedding** 像按照意思找书。即使措辞不同，它也可能发现 `extend a levy` 和 `increase a tax` 意义接近；
- **Hybrid** 把两种分数加权，希望同时保留精确词语和语义相似性。

Embedding 不能自动理解支持、反对、撤销或多议题，因此本 Notebook 只测试“找回来的资料是否更相关”，不生成最终政党立场。

## 怎样避免针对已知案例过拟合？

1. 只使用 validation，不读取 Test 标签；
2. 排除 06d–06g 已经反复检查过的24条 division；
3. 从剩余 validation 中按政策领域确定性分层抽取30条 division；
4. 每条 division 只分配一个政党，形成30个查询，而不是120个重复查询；
5. 人工审核表隐藏检索策略和分数，防止审核者偏向某个方法；
6. 在人工标签完成之前，不宣布任何方法获胜。

本 Notebook 不调用 OpenAI API，不需要 API Key，也不会运行最终 Test。

## 0. First-run setup

Dense Embedding 需要本地 `sentence-transformers`。下面的代码只在缺少该包时执行一次安装。首次运行还会从 Hugging Face 下载一个小型英文模型，之后会使用本地缓存。

为什么选 `all-MiniLM-L6-v2`：

- 当前目标是快速判断 Dense 是否值得加入，而不是寻找最大的模型；
- 知识库只有约5,000个 chunks，小模型足以完成低成本对照；
- 它可以在普通 CPU 上运行；
- 如果 Dense 明显有效，以后再比较更大的模型；如果连基本优势都没有，就没有必要增加复杂度。

预计首次运行需要几分钟，主要时间用于安装、下载和编码。之后读取缓存会快很多。

In [1]:
import importlib.util
import subprocess
import sys

# 如果环境缺少 sentence-transformers，则只安装这一个直接依赖。
INSTALL_IF_MISSING = True
package_missing = importlib.util.find_spec('sentence_transformers') is None
if package_missing and INSTALL_IF_MISSING:
    print('Installing sentence-transformers in the current notebook kernel...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-U', 'sentence-transformers'
    ])
elif package_missing:
    raise ImportError(
        'sentence-transformers is missing. Set INSTALL_IF_MISSING=True and rerun this cell.'
    )
else:
    print('sentence-transformers is already available.')

Installing sentence-transformers in the current notebook kernel...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 1.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 1.6 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 1.4 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sentence-transformers]ence-transformers]


In [2]:
from pathlib import Path
import hashlib
import json
import re
import time

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# 所有输入和输出路径集中在这里，便于检查。
PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / 'processed'
CHUNKS_PATH = PROCESSED_DIR / 'rag_v3' / 'rag_chunks_v3.csv'
OBJECTS_PATH = PROCESSED_DIR / 'policy_object_v6' / 'structured_policy_objects_v6.csv'
KNOWN_AUDIT_PATH = PROCESSED_DIR / 'rag_audit_v3' / 'retrieval_v3_query_metrics.csv'
OUTPUT_DIR = PROCESSED_DIR / 'rag_hybrid_v7'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_VERSION = '06h-dense-hybrid-v7'
RANDOM_STATE = 42
PARTIES = ['conservative', 'green', 'labour', 'liberal-democrat']
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
EMBEDDING_BATCH_SIZE = 32
QUERY_MOTION_MAX_CHARS = 1600
EVALUATION_QUERIES = 30
RESULTS_PER_STRATEGY = 6
MANUAL_TOP_K = 3

# Dense 权重为0就是纯 TF-IDF，为1就是纯 Dense，中间值是 Hybrid。
DENSE_WEIGHTS = [0.00, 0.25, 0.50, 0.75, 1.00]

# 这些是上限，不是强制配额，因此不会为了凑来源而加入零相关证据。
SOURCE_CAPS = {
    'historical_vote': 4,
    'manifesto': 2,
    'bill_reference': 1,
}

print('Notebook version:', NOTEBOOK_VERSION)
print('API calls: 0')
print('LLM calls: 0')
print('Embedding model:', EMBEDDING_MODEL)

Notebook version: 06h-dense-hybrid-v7
API calls: 0
LLM calls: 0
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 1. Load the knowledge base and validation queries

知识库仍然使用经过时间审计的三类资料：manifesto、historical vote 和 Parliament bill reference。06g 的政策对象只是可选提示；如果对象处于 P0、置信度低或只是标题兜底，查询不会使用它，而是依靠 title、legislation 和 motion text。这样可以防止06g的错误控制整个检索。

In [3]:
# 读取已分块知识库和06g的结构化查询信息。
chunks = pd.read_csv(CHUNKS_PATH, low_memory=False)
objects = pd.read_csv(OBJECTS_PATH, low_memory=False)
known_audit = pd.read_csv(KNOWN_AUDIT_PATH, low_memory=False)

chunks['source_date'] = pd.to_datetime(chunks['source_date'], errors='coerce')
objects['motion_date'] = pd.to_datetime(objects['motion_date'], errors='coerce')
chunks['retrieval_text'] = chunks['retrieval_text'].fillna('').astype(str)
chunks['evidence_text'] = chunks['evidence_text'].fillna('').astype(str)
chunks['party'] = chunks['party'].fillna('all').astype(str)

known_audit_keys = set(known_audit['query_division_key'].dropna().astype(str))

print('Knowledge-base chunks:', len(chunks))
print('Known audit divisions excluded:', len(known_audit_keys))
print('Validation divisions available:', int(objects['split'].eq('validation').sum()))
display(chunks['source_type'].value_counts().to_frame('chunks'))

Knowledge-base chunks: 4942
Known audit divisions excluded: 24
Validation divisions available: 200


,chunks
source_type,
historical_vote,3642
manifesto,1170
bill_reference,130


## 2. Select 30 unseen validation queries

这里的30指30个“division + party”查询，不是30×4。每个政党获得7或8个查询。这样仍能比较政党差异，同时把人工审核量控制在合理范围。

抽样不是随手挑选：先固定随机种子，再轮流从不同政策领域取样。相同输入每次会得到完全相同的30条记录。

In [4]:
def safe_text(value):
    # 将缺失值安全转换为空字符串。
    if pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()


def deterministic_domain_sample(frame, n, random_state):
    # 在政策领域之间轮流抽样，防止最大领域占满样本。
    working = frame.copy()
    working['sampling_domain'] = working['policy_domain_primary'].fillna('other_or_unclear')
    rng = np.random.default_rng(random_state)
    domain_rows = {}
    for domain, group in working.groupby('sampling_domain'):
        order = rng.permutation(group.index.to_numpy())
        domain_rows[domain] = list(order)
    domains = sorted(domain_rows, key=lambda key: (-len(domain_rows[key]), key))
    selected = []
    while len(selected) < n and any(domain_rows.values()):
        for domain in domains:
            if domain_rows[domain] and len(selected) < n:
                selected.append(domain_rows[domain].pop(0))
    return working.loc[selected].copy()


candidates = objects[
    objects['split'].eq('validation')
    & ~objects['division_key'].astype(str).isin(known_audit_keys)
    & objects['motion_title_clean'].notna()
    & objects['motion_text_clean'].notna()
    & ~objects['extraction_method'].eq('manual_exclude_multi_issue')
].copy()

if len(candidates) < EVALUATION_QUERIES:
    raise ValueError('Not enough unseen validation divisions for the requested sample.')

evaluation_queries = deterministic_domain_sample(
    candidates, EVALUATION_QUERIES, RANDOM_STATE
).sort_values(['motion_date', 'division_key']).reset_index(drop=True)

# 使用固定轮换并在开始位置上打乱，获得接近均衡的政党数量。
party_cycle = (PARTIES * ((EVALUATION_QUERIES // len(PARTIES)) + 1))[:EVALUATION_QUERIES]
party_rng = np.random.default_rng(RANDOM_STATE + 1)
party_cycle = list(np.array(party_cycle)[party_rng.permutation(EVALUATION_QUERIES)])
evaluation_queries['query_party'] = party_cycle
evaluation_queries['query_id'] = [f'Q{i:03d}' for i in range(1, len(evaluation_queries) + 1)]

def usable_policy_object(row):
    # 只有非低置信度且非P0的对象才能作为辅助提示。
    if row.get('review_priority') == 'P0':
        return ''
    if row.get('extraction_confidence') == 'low':
        return ''
    if row.get('extraction_method') == 'title_fallback':
        return ''
    return safe_text(row.get('canonical_policy_object'))


def build_query_text(row):
    # 标题放在最前面，避免长正文被模型截断时丢失核心主题。
    fields = [
        f"Motion title: {safe_text(row.get('motion_title_clean'))}",
        f"Legislation: {safe_text(row.get('legislation_name_clean'))}",
    ]
    policy_object = usable_policy_object(row)
    if policy_object:
        fields.append(f'Policy object: {policy_object}')
    fields.append(
        f"Motion text: {safe_text(row.get('motion_text_clean'))[:QUERY_MOTION_MAX_CHARS]}"
    )
    return '\n'.join(fields)

evaluation_queries['query_text'] = evaluation_queries.apply(build_query_text, axis=1)

display(evaluation_queries[[
    'query_id', 'division_key', 'motion_date', 'query_party',
    'policy_domain_primary', 'motion_family', 'motion_title_clean',
    'extraction_confidence', 'review_priority',
]])
display(evaluation_queries['query_party'].value_counts().to_frame('queries'))
display(evaluation_queries['policy_domain_primary'].value_counts().to_frame('queries'))

,query_id,division_key,motion_date,query_party,policy_domain_primary,motion_family,motion_title_clean,extraction_confidence,review_priority
0,Q001,pw-2024-01-09-34-commons,2024-01-09,green,health_social_care,amendment_or_clause,NHS Dentistry,medium,P1
1,Q002,pw-2024-01-09-35-commons,2024-01-09,green,immigration_asylum,other_motion,Rwanda Plan Cost and Asylum System,high,P1
2,Q003,pw-2024-01-16-45-commons,2024-01-16,liberal-democrat,constitution_democracy_devolution,bill_stage,Leave for Bill: Scotland (Self-Determination),high,OK
3,Q004,pw-2024-01-22-62-commons,2024-01-22,green,other_or_unclear,bill_stage,Second Reading: Offshore Petroleum Licensing Bill,high,OK
4,Q005,pw-2024-01-30-67-commons,2024-01-30,labour,defence_foreign_affairs,amendment_or_clause,Clause 25 - Sections 20 to 24: further provision,high,P0
5,Q006,pw-2024-01-30-68-commons,2024-01-30,green,NaN,NaN,Clause 50 - Awards of costs,high,OK
6,Q007,pw-2024-02-27-82-commons,2024-02-27,labour,housing_planning_local_government,amendment_or_clause,New Clause 39: — — nsolvency of certain person...,high,OK
7,Q008,pw-2024-02-27-84-commons,2024-02-27,labour,housing_planning_local_government,amendment_or_clause,Amendment: — — nsolvency of certain persons wi...,low,P2
8,Q009,pw-2024-03-01-86-commons,2024-03-01,liberal-democrat,NaN,NaN,Prayers - [Mr Speaker in the Chair]: Conversio...,high,P1
9,Q010,pw-2024-03-18-103-commons,2024-03-18,conservative,immigration_asylum,amendment_or_clause,Disagree: Lords amendment 6,medium,P1


,queries
query_party,
green,8
conservative,8
liberal-democrat,7
labour,7


,queries
policy_domain_primary,
immigration_asylum,3
constitution_democracy_devolution,3
economy_finance_business,3
health_social_care,2
housing_planning_local_government,2
education_skills_children,2
justice_crime_policing,2
digital_data_media_ai,2
transport_infrastructure,2


## 3. Build or load Dense Embeddings

Embedding 把每个 chunk 转成384个数字。相似文本的向量方向更接近，因此可以用 cosine similarity 比较。

参数说明：

- `batch_size=32`：一次处理32段，兼顾普通电脑内存和速度；
- `normalize_embeddings=True`：把每个向量标准化后，向量点积就是 cosine similarity；
- embeddings 会保存到磁盘；chunk顺序或模型改变时才重新计算。

In [5]:
# 使用 chunk ID 和模型名称生成缓存指纹。
chunk_signature = hashlib.sha256(
    ('|'.join(chunks['chunk_id'].astype(str)) + '|' + EMBEDDING_MODEL).encode('utf-8')
).hexdigest()
embedding_path = OUTPUT_DIR / 'chunk_embeddings_v7.npy'
embedding_manifest_path = OUTPUT_DIR / 'embedding_manifest_v7.json'

cache_valid = False
if embedding_path.exists() and embedding_manifest_path.exists():
    with open(embedding_manifest_path, encoding='utf-8') as file:
        embedding_manifest = json.load(file)
    cache_valid = (
        embedding_manifest.get('signature') == chunk_signature
        and embedding_manifest.get('model') == EMBEDDING_MODEL
        and embedding_manifest.get('rows') == len(chunks)
    )

model = SentenceTransformer(EMBEDDING_MODEL)

if cache_valid:
    chunk_embeddings = np.load(embedding_path)
    print('Loaded cached chunk embeddings:', chunk_embeddings.shape)
else:
    start_time = time.perf_counter()
    chunk_embeddings = model.encode(
        chunks['retrieval_text'].tolist(),
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype('float32')
    np.save(embedding_path, chunk_embeddings)
    embedding_manifest = {
        'signature': chunk_signature,
        'model': EMBEDDING_MODEL,
        'rows': int(len(chunks)),
        'dimensions': int(chunk_embeddings.shape[1]),
    }
    with open(embedding_manifest_path, 'w', encoding='utf-8') as file:
        json.dump(embedding_manifest, file, indent=2)
    print('Encoded and cached chunks in seconds:', round(time.perf_counter() - start_time, 1))

query_embeddings = model.encode(
    evaluation_queries['query_text'].tolist(),
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype('float32')

print('Chunk embedding shape:', chunk_embeddings.shape)
print('Query embedding shape:', query_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/155 [00:00<?, ?it/s]

Encoded and cached chunks in seconds: 14.8
Chunk embedding shape: (4942, 384)
Query embedding shape: (30, 384)


## 4. Fair sparse, dense and hybrid scoring

每个查询先执行相同的安全过滤：

- 证据日期必须严格早于查询日期；
- historical vote 和 manifesto 必须属于目标政党，bill reference 可以属于 `all`；
- 删除与当前查询相同的 division；
- 同一 document 最多保留一个 chunk；
- 每种方法使用相同的来源上限。

TF-IDF 参数：

- `ngram_range=(1,2)` 同时使用单词和双词短语；
- `min_df=2` 删除只出现一次的拼写噪声；
- `max_df=0.98` 删除几乎每段都有的通用词；
- `sublinear_tf=True` 降低同一个词反复出现造成的影响；
- `max_features=60000` 限制内存，而不是人工规定6万个政策词。

Hybrid 公式：`score = (1 - dense_weight) × TF-IDF + dense_weight × Dense`。五个权重全部预先指定，不能看完结果后临时增加一个更有利的数字。

In [6]:
def candidate_indices_for_query(row):
    # 返回时间、政党和 division 过滤后的知识库行号。
    query_date = pd.Timestamp(row['motion_date'])
    party = row['query_party']
    allowed_party = chunks['party'].isin([party, 'all'])
    earlier = chunks['source_date'].notna() & chunks['source_date'].lt(query_date)
    different_division = (
        chunks['division_key'].fillna('').astype(str)
        != str(row['division_key'])
    )
    nonempty = chunks['retrieval_text'].str.strip().ne('')
    return np.flatnonzero((allowed_party & earlier & different_division & nonempty).to_numpy())


def sparse_scores_for_query(query_text, candidate_indices):
    # 每个查询只在当时可见的候选语料上拟合 IDF，避免未来文本影响词权重。
    corpus = chunks.iloc[candidate_indices]['retrieval_text'].tolist()
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        norm='l2',
        max_features=60000,
    )
    corpus_matrix = vectorizer.fit_transform(corpus)
    query_vector = vectorizer.transform([query_text])
    return (corpus_matrix @ query_vector.T).toarray().ravel().astype('float32')


def strategy_name(dense_weight):
    # 使用可排序且容易理解的策略名称。
    if dense_weight == 0:
        return 'tfidf_only'
    if dense_weight == 1:
        return 'dense_only'
    return f'hybrid_dense_{int(dense_weight * 100):02d}'


def select_with_source_caps(candidate_frame, score_column, top_k):
    # 上限控制单一来源淹没结果，但不会强制每类来源都出现。
    selected = []
    source_counts = {key: 0 for key in SOURCE_CAPS}
    used_documents = set()
    ordered = candidate_frame.sort_values(score_column, ascending=False)
    for index, row in ordered.iterrows():
        source_type = row['source_type']
        document_id = safe_text(row.get('document_id'))
        if source_type in SOURCE_CAPS:
            if source_counts[source_type] >= SOURCE_CAPS[source_type]:
                continue
        if document_id and document_id in used_documents:
            continue
        selected.append(index)
        if source_type in source_counts:
            source_counts[source_type] += 1
        if document_id:
            used_documents.add(document_id)
        if len(selected) >= top_k:
            break
    return selected


retrieval_rows = []
for query_position, query_row in evaluation_queries.iterrows():
    candidate_indices = candidate_indices_for_query(query_row)
    if len(candidate_indices) == 0:
        continue
    sparse_scores = sparse_scores_for_query(query_row['query_text'], candidate_indices)
    dense_scores = chunk_embeddings[candidate_indices] @ query_embeddings[query_position]
    candidate_frame = chunks.iloc[candidate_indices].copy()
    candidate_frame['tfidf_score'] = sparse_scores
    candidate_frame['dense_score'] = dense_scores.astype('float32')

    for dense_weight in DENSE_WEIGHTS:
        strategy = strategy_name(dense_weight)
        candidate_frame['combined_score'] = (
            (1 - dense_weight) * candidate_frame['tfidf_score']
            + dense_weight * candidate_frame['dense_score']
        )
        selected_indices = select_with_source_caps(
            candidate_frame, 'combined_score', RESULTS_PER_STRATEGY
        )
        for rank, selected_index in enumerate(selected_indices, start=1):
            evidence = candidate_frame.loc[selected_index]
            retrieval_rows.append({
                'query_id': query_row['query_id'],
                'query_division_key': query_row['division_key'],
                'query_party': query_row['query_party'],
                'query_date': query_row['motion_date'],
                'query_title': query_row['motion_title_clean'],
                'query_policy_domain': query_row['policy_domain_primary'],
                'query_motion_family': query_row['motion_family'],
                'query_text': query_row['query_text'],
                'strategy': strategy,
                'dense_weight': dense_weight,
                'rank': rank,
                'tfidf_score': float(evidence['tfidf_score']),
                'dense_score': float(evidence['dense_score']),
                'combined_score': float(evidence['combined_score']),
                'chunk_id': evidence['chunk_id'],
                'document_id': evidence['document_id'],
                'source_type': evidence['source_type'],
                'evidence_party': evidence['party'],
                'evidence_date': evidence['source_date'],
                'evidence_division_key': evidence['division_key'],
                'evidence_title': evidence['title'],
                'evidence_policy_domain': evidence['policy_domain'],
                'stance_label': evidence['stance_label'],
                'source_url': evidence['source_url'],
                'page_number': evidence['page_number'],
                'evidence_text': evidence['evidence_text'],
            })

retrieval_results = pd.DataFrame(retrieval_rows)
print('Retrieval result rows:', len(retrieval_results))
display(pd.crosstab(retrieval_results['strategy'], retrieval_results['source_type']))

Retrieval result rows: 800


source_type,bill_reference,historical_vote,manifesto
strategy,,,
dense_only,30,120,10
hybrid_dense_25,30,120,10
hybrid_dense_50,30,120,10
hybrid_dense_75,30,120,10
tfidf_only,30,120,10


## 5. Structural safety checks

这些检查只能证明流程没有泄漏或重复，不能证明检索结果语义正确。语义质量必须由盲审标签决定。

In [7]:
# 检查时间泄漏、相同 division、重复 chunk 和已知审计样本污染。
retrieval_results['query_date'] = pd.to_datetime(retrieval_results['query_date'])
retrieval_results['evidence_date'] = pd.to_datetime(retrieval_results['evidence_date'])

duplicate_counts = (
    retrieval_results.groupby(['query_id', 'strategy'])['chunk_id']
    .apply(lambda values: values.duplicated().sum())
)
structural_gates = {
    'evaluation_query_count_gate': len(evaluation_queries) == EVALUATION_QUERIES,
    'known_audit_exclusion_gate': not evaluation_queries['division_key'].isin(known_audit_keys).any(),
    'strict_temporal_gate': bool(
        retrieval_results['evidence_date'].lt(retrieval_results['query_date']).all()
    ),
    'same_division_gate': not (
        retrieval_results['query_division_key'].astype(str)
        == retrieval_results['evidence_division_key'].fillna('').astype(str)
    ).any(),
    'duplicate_chunk_gate': int(duplicate_counts.sum()) == 0,
    'maximum_results_gate': int(
        retrieval_results.groupby(['query_id', 'strategy']).size().max()
    ) <= RESULTS_PER_STRATEGY,
    'all_parties_represented_gate': set(evaluation_queries['query_party']) == set(PARTIES),
    'no_test_rows_gate': evaluation_queries['split'].eq('validation').all(),
}
display(pd.Series(structural_gates, name='passed').to_frame())

,passed
evaluation_query_count_gate,True
known_audit_exclusion_gate,True
strict_temporal_gate,True
same_division_gate,True
duplicate_chunk_gate,True
maximum_results_gate,True
all_parties_represented_gate,True
no_test_rows_gate,True


## 6. Create a blinded manual-review queue

每种策略只取前三条证据进入人工比较。不同策略返回同一个 chunk 时只审核一次。审核文件故意不包含 strategy、rank 和 score。

人工角色定义：

- `directional_direct`：证据直接谈论同一政策，而且可以支持方向判断；
- `relevant_context`：同一政策领域或相近工具，但不能直接证明政党方向；
- `background_reference`：主要提供 Bill 或制度背景；
- `reject`：主题相似但政策不同，或基本无关。

不要根据语言是否听起来合理来判断，而要问：这条证据能否直接帮助回答当前政党对当前政策的态度？

In [8]:
# 合并各策略的前三名，并为每个查询-证据组合生成稳定 ID。
manual_candidates = retrieval_results[retrieval_results['rank'].le(MANUAL_TOP_K)].copy()
manual_candidates['candidate_id'] = manual_candidates.apply(
    lambda row: hashlib.sha1(
        f"{row['query_id']}|{row['chunk_id']}".encode('utf-8')
    ).hexdigest()[:16],
    axis=1,
)

strategy_map = manual_candidates[[
    'candidate_id', 'query_id', 'chunk_id', 'strategy', 'dense_weight',
    'rank', 'tfidf_score', 'dense_score', 'combined_score',
]].sort_values(['query_id', 'candidate_id', 'strategy'])

blinded_queue = (
    manual_candidates.sort_values(['query_id', 'candidate_id'])
    .drop_duplicates(['query_id', 'chunk_id'])
)
blinded_queue['manual_role'] = ''
blinded_queue['manual_notes'] = ''

blinded_columns = [
    'candidate_id', 'query_id', 'query_division_key', 'query_party',
    'query_date', 'query_title', 'query_policy_domain', 'query_motion_family',
    'query_text', 'source_type', 'evidence_party', 'evidence_date',
    'evidence_title', 'evidence_policy_domain', 'stance_label',
    'source_url', 'page_number', 'evidence_text', 'manual_role', 'manual_notes',
]
blinded_queue = blinded_queue[blinded_columns]

BLINDED_QUEUE_PATH = OUTPUT_DIR / 'manual_hybrid_review_queue_v7.csv'
COMPLETED_LABELS_PATH = OUTPUT_DIR / 'manual_hybrid_review_completed_v7.csv'
STRATEGY_MAP_PATH = OUTPUT_DIR / 'candidate_strategy_map_v7.csv'

# 每次都刷新空白模板，但绝不覆盖已完成的人工标签文件。
blinded_queue.to_csv(BLINDED_QUEUE_PATH, index=False)
strategy_map.to_csv(STRATEGY_MAP_PATH, index=False)

print('Evaluation queries:', blinded_queue['query_id'].nunique())
print('Unique evidence rows for blind review:', len(blinded_queue))
print('Blinded queue:', BLINDED_QUEUE_PATH)
print('Completed-label path:', COMPLETED_LABELS_PATH)
display(blinded_queue.groupby('query_party').size().to_frame('evidence_rows'))
display(blinded_queue.head(12))

Evaluation queries: 30
Unique evidence rows for blind review: 183
Blinded queue: /Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_hybrid_v7/manual_hybrid_review_queue_v7.csv
Completed-label path: /Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_hybrid_v7/manual_hybrid_review_completed_v7.csv


,evidence_rows
query_party,
conservative,46
green,49
labour,45
liberal-democrat,43


,candidate_id,query_id,query_division_key,query_party,query_date,query_title,query_policy_domain,query_motion_family,query_text,source_type,evidence_party,evidence_date,evidence_title,evidence_policy_domain,stance_label,source_url,page_number,evidence_text,manual_role,manual_notes
2,00d50b4e4cf83195,Q001,pw-2024-01-09-34-commons,green,2024-01-09,NHS Dentistry,health_social_care,amendment_or_clause,Motion title: NHS Dentistry\nLegislation: \nPo...,historical_vote,green,2020-01-29,Policing and Crime,justice_crime_policing,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
0,3a92850c5127aa19,Q001,pw-2024-01-09-34-commons,green,2024-01-09,NHS Dentistry,health_social_care,amendment_or_clause,Motion title: NHS Dentistry\nLegislation: \nPo...,historical_vote,green,2023-01-11,NHS: Long-term Strategy,health_social_care,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
22,3d79bb7ba7b91641,Q001,pw-2024-01-09-34-commons,green,2024-01-09,NHS Dentistry,health_social_care,amendment_or_clause,Motion title: NHS Dentistry\nLegislation: \nPo...,historical_vote,green,2016-11-16,"AUTUMN STATEMENT DISTRIBUTIONAL ANALYSIS, UNIV...",economy_finance_business,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
1,7b4dfe9ba5065237,Q001,pw-2024-01-09-34-commons,green,2024-01-09,NHS Dentistry,health_social_care,amendment_or_clause,Motion title: NHS Dentistry\nLegislation: \nPo...,historical_vote,green,2020-03-04,Health Inequalities,health_social_care,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
7,b955744d7c26a731,Q001,pw-2024-01-09-34-commons,green,2024-01-09,NHS Dentistry,health_social_care,amendment_or_clause,Motion title: NHS Dentistry\nLegislation: \nPo...,historical_vote,green,2023-06-07,Mental Health Treatment and Support,health_social_care,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
26,1aa0f931af1bc857,Q002,pw-2024-01-09-35-commons,green,2024-01-09,Rwanda Plan Cost and Asylum System,immigration_asylum,other_motion,Motion title: Rwanda Plan Cost and Asylum Syst...,historical_vote,green,2019-01-21,HEALTHCARE (INTERNATIONAL ARRANGEMENTS) BILL: ...,defence_foreign_affairs,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
27,4e67bf5a5990e051,Q002,pw-2024-01-09-35-commons,green,2024-01-09,Rwanda Plan Cost and Asylum System,immigration_asylum,other_motion,Motion title: Rwanda Plan Cost and Asylum Syst...,historical_vote,green,2021-05-19,Amendment: A Plan for the NHS and Social Care,health_social_care,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
36,83edd58ca93ca001,Q002,pw-2024-01-09-35-commons,green,2024-01-09,Rwanda Plan Cost and Asylum System,immigration_asylum,other_motion,Motion title: Rwanda Plan Cost and Asylum Syst...,historical_vote,green,2016-11-16,"AUTUMN STATEMENT DISTRIBUTIONAL ANALYSIS, UNIV...",economy_finance_business,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
37,98de0172592dcaef,Q002,pw-2024-01-09-35-commons,green,2024-01-09,Rwanda Plan Cost and Asylum System,immigration_asylum,other_motion,Motion title: Rwanda Plan Cost and Asylum Syst...,historical_vote,green,2021-09-21,Working People — — Finances: Government Policy,economy_finance_business,support,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,
30,99c01ff5720b9617,Q002,pw-2024-01-09-35-commons,green,2024-01-09,Rwanda Plan Cost and Asylum System,immigration_asylum,other_motion,Motion title: Rwanda Plan Cost and Asylum Syst...,historical_vote,green,2023-12-12,Second Reading: Safety of Rwanda (Asylum and I...,immigration_asylum,oppose,NaN,NaN,Historical parliamentary vote.\nParty: green.\...,,


## 7. Evaluate completed manual labels when available

第一次运行时这里会显示 `pending_manual_review`。不要手动挑选看起来最好的权重。等 Codex 完成盲审并保存 `manual_hybrid_review_completed_v7.csv` 后，重新运行本节和最后摘要。

主要指标：

- `direct_precision_at_3`：前三条中真正直接证据的比例；
- `useful_precision_at_3`：前三条中直接证据或相关上下文的比例；
- `direct_query_coverage`：30个查询中至少找到一条直接证据的比例；
- `reject_rate_at_3`：前三条被判定为无关的比例。

采用 Hybrid 的条件不是“比 TF-IDF 高一点”，而是直接证据精度至少比两个单模型中更好的一个高0.05，并且 direct coverage 不下降。

In [9]:
VALID_MANUAL_ROLES = {
    'directional_direct', 'relevant_context',
    'background_reference', 'reject',
}

evaluation_status = 'pending_manual_review'
strategy_metrics = pd.DataFrame()
selected_strategy = None
hybrid_acceptance = False
completed_label_rows = 0

if COMPLETED_LABELS_PATH.exists():
    completed = pd.read_csv(COMPLETED_LABELS_PATH, low_memory=False)
    completed['manual_role'] = completed['manual_role'].fillna('').astype(str).str.strip()
    invalid_roles = set(completed.loc[
        completed['manual_role'].ne(''), 'manual_role'
    ]) - VALID_MANUAL_ROLES
    if invalid_roles:
        raise ValueError(f'Invalid manual roles: {sorted(invalid_roles)}')
    completed_label_rows = int(completed['manual_role'].isin(VALID_MANUAL_ROLES).sum())
    fully_labeled = (
        len(completed) == len(blinded_queue)
        and completed['manual_role'].isin(VALID_MANUAL_ROLES).all()
        and completed['candidate_id'].is_unique
    )
    if fully_labeled:
        labels = completed[['candidate_id', 'manual_role']]
        scored = strategy_map.merge(
            labels, on='candidate_id', how='left', validate='many_to_one'
        )
        scored['is_direct'] = scored['manual_role'].eq('directional_direct')
        scored['is_useful'] = scored['manual_role'].isin([
            'directional_direct', 'relevant_context'
        ])
        scored['is_reject'] = scored['manual_role'].eq('reject')

        metric_rows = []
        for strategy, group in scored.groupby('strategy'):
            top = group[group['rank'].le(MANUAL_TOP_K)].copy()
            query_direct = top.groupby('query_id')['is_direct'].any()
            metric_rows.append({
                'strategy': strategy,
                'dense_weight': float(top['dense_weight'].iloc[0]),
                'evidence_rows': len(top),
                'direct_precision_at_3': float(top['is_direct'].mean()),
                'useful_precision_at_3': float(top['is_useful'].mean()),
                'reject_rate_at_3': float(top['is_reject'].mean()),
                'direct_query_coverage': float(query_direct.mean()),
            })
        strategy_metrics = pd.DataFrame(metric_rows).sort_values(
            ['direct_precision_at_3', 'direct_query_coverage', 'useful_precision_at_3'],
            ascending=False,
        )
        selected_strategy = strategy_metrics.iloc[0]['strategy']
        best_baseline_precision = strategy_metrics[
            strategy_metrics['strategy'].isin(['tfidf_only', 'dense_only'])
        ]['direct_precision_at_3'].max()
        selected_row = strategy_metrics.iloc[0]
        baseline_coverage = strategy_metrics[
            strategy_metrics['strategy'].isin(['tfidf_only', 'dense_only'])
        ]['direct_query_coverage'].max()
        hybrid_acceptance = bool(
            str(selected_strategy).startswith('hybrid_')
            and selected_row['direct_precision_at_3'] >= best_baseline_precision + 0.05
            and selected_row['direct_query_coverage'] >= baseline_coverage
        )
        evaluation_status = 'manual_evaluation_complete'
        strategy_metrics.to_csv(OUTPUT_DIR / 'strategy_metrics_v7.csv', index=False)
        display(strategy_metrics.round(3))
    else:
        print('Completed-label file exists but is incomplete.')
else:
    print('Manual labels are not available yet. This is expected on the first run.')

Manual labels are not available yet. This is expected on the first run.


## 8. Save outputs and print the review summary

第一次运行后，把最后摘要发给 Codex。Codex 会审核盲审队列，而不是要求你阅读所有英文证据。完成标签后再重新运行第7和第8节。

In [10]:
# 保存抽样、完整检索结果和运行清单。
evaluation_queries.to_csv(OUTPUT_DIR / 'evaluation_queries_v7.csv', index=False)
retrieval_results.to_csv(OUTPUT_DIR / 'retrieval_results_all_strategies_v7.csv', index=False)

run_manifest = {
    'notebook_version': NOTEBOOK_VERSION,
    'api_calls': 0,
    'llm_calls': 0,
    'embedding_model': EMBEDDING_MODEL,
    'knowledge_base_chunks': int(len(chunks)),
    'evaluation_queries': int(len(evaluation_queries)),
    'known_audit_divisions_excluded': int(len(known_audit_keys)),
    'dense_weights': DENSE_WEIGHTS,
    'structural_gates': {key: bool(value) for key, value in structural_gates.items()},
    'evaluation_status': evaluation_status,
    'completed_label_rows': completed_label_rows,
    'selected_strategy': selected_strategy,
    'hybrid_acceptance': bool(hybrid_acceptance),
}
with open(OUTPUT_DIR / 'run_manifest_v7.json', 'w', encoding='utf-8') as file:
    json.dump(run_manifest, file, ensure_ascii=False, indent=2)

print('=== DENSE HYBRID RETRIEVAL SUMMARY FOR REVIEW ===')
print('Notebook version:', NOTEBOOK_VERSION)
print('API calls: 0')
print('LLM calls: 0')
print('Embedding model:', EMBEDDING_MODEL)
print('Knowledge-base chunks:', len(chunks))
print('Evaluation queries:', len(evaluation_queries))
print('Known audit divisions excluded:', len(known_audit_keys))
print('Query party counts:')
print(evaluation_queries['query_party'].value_counts().sort_index().to_string())
print('Dense weights tested:', DENSE_WEIGHTS)
print('Results per strategy:', RESULTS_PER_STRATEGY)
print('Manual review top K:', MANUAL_TOP_K)
print('Blind-review evidence rows:', len(blinded_queue))
print('Completed manual-label rows:', completed_label_rows)
print('Structural gates:', structural_gates)
print('Evaluation status:', evaluation_status)
print('Selected strategy:', selected_strategy)
print('Hybrid acceptance:', hybrid_acceptance)
if not strategy_metrics.empty:
    print('Strategy metrics:')
    print(strategy_metrics.round(3).to_string(index=False))
print('Output directory:', OUTPUT_DIR)
if evaluation_status == 'pending_manual_review':
    print('Next step: Codex reviews the blinded evidence queue; do not start 07.')
else:
    print('Next step: decide whether the measured Hybrid gain justifies rebuilding RAG.')
print('=== END DENSE HYBRID RETRIEVAL SUMMARY ===')

=== DENSE HYBRID RETRIEVAL SUMMARY FOR REVIEW ===
Notebook version: 06h-dense-hybrid-v7
API calls: 0
LLM calls: 0
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Knowledge-base chunks: 4942
Evaluation queries: 30
Known audit divisions excluded: 24
Query party counts:
query_party
conservative        8
green               8
labour              7
liberal-democrat    7
Dense weights tested: [0.0, 0.25, 0.5, 0.75, 1.0]
Results per strategy: 6
Manual review top K: 3
Blind-review evidence rows: 183
Completed manual-label rows: 0
Structural gates: {'evaluation_query_count_gate': True, 'known_audit_exclusion_gate': True, 'strict_temporal_gate': True, 'same_division_gate': True, 'duplicate_chunk_gate': True, 'maximum_results_gate': True, 'all_parties_represented_gate': True, 'no_test_rows_gate': True}
Evaluation status: pending_manual_review
Selected strategy: None
Hybrid acceptance: False
Output directory: /Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_hybrid_v7
Next st